In [ ]:
%%capture
%pip install transformers datasets evaluate sacrebleu torch huggingface_hub
%pip install accelerate sentencepiece tiktoken peft bitsandbytes safetensors trl

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import torch

print(torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

Tesla T4
BF16 supported: True


In [ ]:
from datasets import load_dataset

data = load_dataset("kalixlouiis/Myanmar-English-general-text-translation")

In [ ]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
model_checkpoint = "Ko-Yin-Maung/mig-burmese-llm"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
def tokenize(examples):
    prefix = "<start_of_turn>user\nTranslate to English: "
    suffix = "\n<end_of_turn><start_of_turn>model\n"
    prompt = prefix + examples["my"] + suffix + examples["en"] + "<end_of_turn>\n"
    return tokenizer(
        prompt,
        truncation=True,
        max_length=128,
        padding=False
    )

tokenized_train_data = data['train'].map(tokenize)
tokenized_val_data = data['validation'].map(tokenize)
tokenized_test_data = data['test'].map(tokenize)

Map:   0%|          | 0/9616 [00:00<?, ? examples/s]

Map:   0%|          | 0/1202 [00:00<?, ? examples/s]

Map:   0%|          | 0/1203 [00:00<?, ? examples/s]

In [ ]:
tokenized_train_data = tokenized_train_data.remove_columns(["en", "my"])
tokenized_val_data = tokenized_val_data.remove_columns(["en", "my"])
tokenized_test_data = tokenized_test_data.remove_columns(["en", "my"])

In [ ]:
import torch
from torch.utils.data import Subset

indices = list(range(1000))
tokenized_train_data = Subset(tokenized_train_data, list(range(100)))
tokenized_val_data = Subset(tokenized_val_data, list(range(10)))
tokenized_test_data = Subset(tokenized_test_data, list(range(10)))

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained(
   model_checkpoint,
   device_map="cuda:0",
   dtype=torch.bfloat16,
)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [ ]:
batch = data_collator([
    tokenized_val_data[i]
    for i in range(8)
])

batch = {k: v.to(model.device) for k, v in batch.items()}

model.eval()

with torch.no_grad():
    outputs = model(**batch)

print("dtype:", model.dtype)
print("loss:", outputs.loss.item())
print("loss finite:", torch.isfinite(outputs.loss).item())
print("logits finite:", torch.isfinite(outputs.logits).all().item())

dtype: torch.bfloat16
loss: 7.766972541809082
loss finite: True
logits finite: True


In [ ]:
training_args = TrainingArguments(
    output_dir="burmese-english-model",
    eval_strategy="epoch",
    learning_rate=3e-5,
    weight_decay=0.01,
    push_to_hub=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=True,
    fp16=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_data,
    eval_dataset=tokenized_val_data,
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,2.421539
2,No log,2.569166
3,No log,2.945997


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=300, training_loss=1.5583973185221354, metrics={'train_runtime': 217.8276, 'train_samples_per_second': 1.377, 'train_steps_per_second': 1.377, 'total_flos': 68903278398720.0, 'train_loss': 1.5583973185221354, 'epoch': 3.0})

In [ ]:
query = "ဒီလကုန်ကျရင် စာမေးပွဲရှိတယ်"
prompt = f"<start_of_turn>user\nTranslate to English: {query}\n<end_of_turn><start_of_turn>model\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, temperature=0.01)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
Translate to English: ဒီလကုန်ကျရင် စာမေးပွဲရှိတယ်
model
There is a question mark on the exam.
